Notebook to showcase the BERT model results


1. BERT model - Fine Tuned in LIAR, tested in LIAR


In [ ]:
import pandas as pd
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertTokenizer, BertForSequenceClassification
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix

test_df = pd.read_csv("./LIAR dataset/test modified.tsv", sep="\t")

test_df = test_df[['statement', 'label']]

def map_label(label):
    return 1 if label in ['TRUE', 'mostly-true'] else 0

test_df['label'] = test_df['label'].apply(map_label)

print(test_df.head())

                                           statement  label
0  Building a wall on the U.S.-Mexico border will...      1
1  Wisconsin is on pace to double the number of l...      0
2  Says John McCain has done nothing to help the ...      0
3  Suzanne Bonamici supports a plan that will cut...      0
4  When asked by a reporter whether hes at the ce...      0


In [ ]:
tokenizer = BertTokenizer.from_pretrained('./bert_liar_model')
model = BertForSequenceClassification.from_pretrained('./bert_liar_model')

model.eval()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
encodings = tokenizer(
    list(test_df['statement']), 
    padding=True, 
    truncation=True, 
    max_length=512, 
    return_tensors='pt'
)

dataset = TensorDataset(encodings['input_ids'], encodings['attention_mask'], torch.tensor(test_df['label'].values))
batch_size = 32 

test_loader = DataLoader(dataset, batch_size=batch_size)


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, f1_score
import numpy as np

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        input_ids, attention_mask, labels = [b.to(device) for b in batch]

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)


In [ ]:
accuracy = accuracy_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 0.6701
Recall: 0.4744
Precision: 0.5392
Specificity: 0.7775
F1 Score: 0.5047


2. BERT model - Fine tuned in LIAR, tested in ReNew (testing dataset)


In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch

tokenizer = BertTokenizer.from_pretrained("./bert_liar_model")
model = BertForSequenceClassification.from_pretrained("./bert_liar_model")
model.eval()


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e

In [ ]:
import pandas as pd

test_file_path = "./ReNew dataset/combined renew test.csv"

df_test = pd.read_csv(test_file_path)

assert 'statement' in df_test.columns and 'label' in df_test.columns, "Missing 'statement' or 'label' column."

print(df_test.shape)
df_test.head()


(527, 2)


,label,statement
0,1,Evidence is mounting that marijuana legalizati...
1,1,We want to be very clear about what our positi...
2,1,Very close to 160 million people are now worki...
3,0,It's a question of fairness. Should a waitress...
4,1,Matt Gaetz defends lone no vote on anti-human ...


In [ ]:
inputs = tokenizer(
    df_test['statement'].tolist(),
    padding=True,
    truncation=True,
    return_tensors="pt"
)

from torch.utils.data import DataLoader, TensorDataset

labels = torch.tensor(df_test['label'].values)

dataset = TensorDataset(inputs['input_ids'], inputs['attention_mask'], labels)
dataloader = DataLoader(dataset, batch_size=16)


In [ ]:
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in dataloader:
        input_ids, attention_mask, batch_labels = batch
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.tolist())
        all_labels.extend(batch_labels.tolist())


In [ ]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, confusion_matrix, f1_score
import numpy as np

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

accuracy = accuracy_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
specificity = tn / (tn + fp)

print(f"Accuracy: {accuracy:.4f}")
print(f"Recall: {recall:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"F1 Score: {f1:.4f}")


Accuracy: 0.6186
Recall: 0.3333
Precision: 0.7788
Specificity: 0.9049
F1 Score: 0.4668


3. BERT model - Fine Tuned in LIAR + R, tested in LIAR


In [ ]:
import pandas as pd
from transformers import BertForSequenceClassification, AutoTokenizer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
from torch.utils.data import DataLoader

liar_renew_ft_model_path = "./fine tuned models/bert_renew_combined_finetuned_model"
liar_renew_ft_tokenizer = AutoTokenizer.from_pretrained(liar_renew_ft_model_path)
liar_renew_ft_model = BertForSequenceClassification.from_pretrained(liar_renew_ft_model_path)

liar_renew_ft_df = pd.read_csv('./LIAR dataset/cleaned_test.csv')

liar_renew_ft_encodings = liar_renew_ft_tokenizer(
    liar_renew_ft_df['statement'].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

liar_renew_ft_labels = liar_renew_ft_df['label'].tolist()

class LiarRenewFTDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

liar_renew_ft_dataset = LiarRenewFTDataset(liar_renew_ft_encodings, liar_renew_ft_labels)
liar_renew_ft_loader = DataLoader(liar_renew_ft_dataset, batch_size=64)

liar_renew_ft_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
liar_renew_ft_model.to(device)

liar_renew_ft_preds = []

with torch.no_grad():
    for batch in liar_renew_ft_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = liar_renew_ft_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        liar_renew_ft_preds.extend(preds)

liar_renew_ft_accuracy = accuracy_score(liar_renew_ft_labels, liar_renew_ft_preds)
liar_renew_ft_precision = precision_score(liar_renew_ft_labels, liar_renew_ft_preds)
liar_renew_ft_recall = recall_score(liar_renew_ft_labels, liar_renew_ft_preds)
liar_renew_ft_f1 = f1_score(liar_renew_ft_labels, liar_renew_ft_preds)
tn, fp, fn, tp = confusion_matrix(liar_renew_ft_labels, liar_renew_ft_preds).ravel()
liar_renew_ft_specificity = tn / (tn + fp)

print("=== Evaluation: LIAR Test Set Using liar_renew_ft_model ===")
print(f"Accuracy   : {liar_renew_ft_accuracy:.4f}")
print(f"Precision  : {liar_renew_ft_precision:.4f}")
print(f"Recall     : {liar_renew_ft_recall:.4f}")
print(f"F1 Score   : {liar_renew_ft_f1:.4f}")
print(f"Specificity: {liar_renew_ft_specificity:.4f}")


=== Evaluation: LIAR Test Set Using liar_renew_ft_model ===
Accuracy   : 0.6646
Precision  : 0.5248
Recall     : 0.5657
F1 Score   : 0.5445
Specificity: 0.7188


4. BERT model - Fine Tuned in LIAR + R, tested in ReNew Test


In [ ]:
import pandas as pd
from transformers import BertForSequenceClassification, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch

df_ft2_model_path = "./fine tuned models/bert_renew_combined_finetuned_model"
df_ft2_tokenizer = AutoTokenizer.from_pretrained(df_ft2_model_path)
df_ft2_model = BertForSequenceClassification.from_pretrained(df_ft2_model_path)

df_ft2_test_df = pd.read_csv('./ReNew dataset/combined renew test.csv')
df_ft2_test_labels = df_ft2_test_df['label'].tolist()

df_ft2_test_encodings = df_ft2_tokenizer(
    df_ft2_test_df['statement'].tolist(),
    truncation=True,
    padding=True,
    max_length=128
)

class df_ft2_TestDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

df_ft2_test_dataset = df_ft2_TestDataset(df_ft2_test_encodings, df_ft2_test_labels)
df_ft2_test_loader = DataLoader(df_ft2_test_dataset, batch_size=64)

df_ft2_model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
df_ft2_model.to(device)

df_ft2_preds = []
with torch.no_grad():
    for batch in df_ft2_test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        outputs = df_ft2_model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        df_ft2_preds.extend(preds)

accuracy = accuracy_score(df_ft2_test_labels, df_ft2_preds)
precision = precision_score(df_ft2_test_labels, df_ft2_preds)
recall = recall_score(df_ft2_test_labels, df_ft2_preds)
f1 = f1_score(df_ft2_test_labels, df_ft2_preds)
tn, fp, fn, tp = confusion_matrix(df_ft2_test_labels, df_ft2_preds).ravel()
specificity = tn / (tn + fp)

print("=== Evaluation on Combined Renew Test Set ===")
print(f"Accuracy   : {accuracy:.4f}")
print(f"Precision  : {precision:.4f}")
print(f"Recall     : {recall:.4f}")
print(f"F1 Score   : {f1:.4f}")
print(f"Specificity: {specificity:.4f}")


=== Evaluation on Combined Renew Test Set ===
Accuracy   : 0.7913
Precision  : 0.8532
Recall     : 0.7045
F1 Score   : 0.7718
Specificity: 0.8783
